In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# TEAM 3 - PHASE 4 INCREMENTAL SILVER VALIDATION & CLEANSING
# - Detect newest _batch_id already loaded into Bronze
# - Validate only that Bronze batch
# - Use existing Silver/reference rows only for referential checks
# - MERGE accepted/audit rows into existing Silver tables
# - Recompute delivery totals/flags only for order-lines affected by
#   delivery events in the current batch
# ============================================================

dbutils.widgets.text("catalog", "dbwteam3")
dbutils.widgets.text("schema", "order_fulfillment")
dbutils.widgets.text("input_batch", "latest")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
INPUT_BATCH = dbutils.widgets.get("input_batch").strip().lower()
TARGET = f"{CATALOG}.{SCHEMA}"

if not CATALOG or not SCHEMA or not INPUT_BATCH:
    raise ValueError("catalog, schema and input_batch must not be empty")

BRONZE = {
    "pharmacies": f"{TARGET}.bronze_pharmacies",
    "orders": f"{TARGET}.bronze_orders",
    "order_lines": f"{TARGET}.bronze_order_lines",
    "shipment_lines": f"{TARGET}.bronze_shipment_lines",
    "delivery_events": f"{TARGET}.bronze_delivery_events",
}

SILVER = {
    "pharmacies": f"{TARGET}.silver_pharmacies",
    "orders": f"{TARGET}.silver_orders",
    "order_lines": f"{TARGET}.silver_order_lines",
    "shipment_lines": f"{TARGET}.silver_shipment_lines",
    "delivery_events": f"{TARGET}.silver_delivery_events",
    "rejected_orders": f"{TARGET}.silver_rejected_orders",
    "rejected_order_lines": f"{TARGET}.silver_rejected_order_lines",
    "rejected_delivery_events": f"{TARGET}.silver_rejected_delivery_events",
    "duplicate_delivery_events": f"{TARGET}.silver_duplicate_delivery_events",
    "quarantined_delivery_events": f"{TARGET}.silver_quarantined_delivery_events",
}

for name, table in BRONZE.items():
    if not spark.catalog.tableExists(table):
        raise ValueError(f"Required Bronze table is missing: {table}")

for name, table in SILVER.items():
    if not spark.catalog.tableExists(table):
        raise ValueError(
            f"Required Silver table is missing: {table}. "
            "Run the existing full-refresh Silver notebook once before switching to this incremental notebook."
        )

# -----------------------------------------------------------------
# 1. Resolve the batch.
# We detect from the four transactional Bronze tables, not pharmacies,
# because pharmacies are reference data and may remain in an older batch.
# -----------------------------------------------------------------
transactional_bronze_tables = [
    BRONZE["orders"],
    BRONZE["order_lines"],
    BRONZE["shipment_lines"],
    BRONZE["delivery_events"],
]

batch_union = None
for table in transactional_bronze_tables:
    df = (
        spark.table(table)
        .select(F.col("_batch_id").cast("string").alias("_batch_id"))
        .filter(F.col("_batch_id").rlike(r"^batch_[0-9]{3}$"))
        .distinct()
    )
    batch_union = df if batch_union is None else batch_union.unionByName(df)

available_batches = [r["_batch_id"] for r in batch_union.distinct().collect()]
if not available_batches:
    raise ValueError("No valid batch_NNN values were found in transactional Bronze tables")

if INPUT_BATCH == "latest":
    BATCH_ID = max(available_batches, key=lambda x: int(x.split("_")[1]))
elif INPUT_BATCH.startswith("batch_") and INPUT_BATCH in available_batches:
    BATCH_ID = INPUT_BATCH
else:
    raise ValueError(
        f"input_batch must be 'latest' or an existing batch_NNN. Found: {INPUT_BATCH}"
    )

print("=" * 70)
print("PHASE 4 - INCREMENTAL SILVER VALIDATION")
print("Target   :", TARGET)
print("Batch    :", BATCH_ID)
print("=" * 70)

# -----------------------------------------------------------------
# 2. Read ONLY current-batch Bronze transaction rows.
# Full/reference tables are read only as lookup keys where required.
# -----------------------------------------------------------------
pharmacies_lookup = spark.table(BRONZE["pharmacies"])
orders_bronze = spark.table(BRONZE["orders"]).filter(F.col("_batch_id") == BATCH_ID)
order_lines_bronze = spark.table(BRONZE["order_lines"]).filter(F.col("_batch_id") == BATCH_ID)
shipment_lines_bronze = spark.table(BRONZE["shipment_lines"]).filter(F.col("_batch_id") == BATCH_ID)
delivery_events_bronze = spark.table(BRONZE["delivery_events"]).filter(F.col("_batch_id") == BATCH_ID)

input_counts = {
    "orders": orders_bronze.count(),
    "order_lines": order_lines_bronze.count(),
    "shipment_lines": shipment_lines_bronze.count(),
    "delivery_events": delivery_events_bronze.count(),
}
print("Current Bronze batch input counts:", input_counts)

# Existing accepted Silver keys are necessary because a new batch may legitimately
# reference an order/order-line/shipment that was accepted in a prior batch.
existing_order_ids = (
    spark.table(SILVER["orders"])
    .select(F.trim("order_id").alias("order_id"))
    .filter(F.col("order_id").isNotNull() & (F.col("order_id") != ""))
    .distinct()
)
existing_order_line_keys = (
    spark.table(SILVER["order_lines"])
    .select(F.trim("order_id").alias("order_id"), F.trim("line_id").alias("line_id"))
    .distinct()
)
existing_shipment_keys = (
    spark.table(SILVER["shipment_lines"])
    .select(
        F.trim("shipment_id").alias("shipment_id"),
        F.trim("order_id").alias("order_id"),
        F.trim("line_id").alias("line_id"),
    )
    .distinct()
)

# -----------------------------------------------------------------
# 3. Validate CURRENT-BATCH orders.
# -----------------------------------------------------------------
valid_pharmacy_ids = (
    pharmacies_lookup
    .select(F.trim("pharmacy_id").alias("pharmacy_id"))
    .filter(F.col("pharmacy_id").isNotNull() & (F.col("pharmacy_id") != ""))
    .distinct()
)

orders_checked = (
    orders_bronze
    .withColumn("_order_id_clean", F.trim("order_id"))
    .withColumn("_pharmacy_id_clean", F.trim("pharmacy_id"))
    .withColumn("order_time", F.expr("try_cast(order_time AS TIMESTAMP)"))
    .withColumn("promised_delivery_time", F.expr("try_cast(promised_delivery_time AS TIMESTAMP)"))
    .join(
        valid_pharmacy_ids.withColumnRenamed("pharmacy_id", "_known_pharmacy_id"),
        F.col("_pharmacy_id_clean") == F.col("_known_pharmacy_id"),
        "left",
    )
    .withColumn(
        "validation_reason",
        F.when(F.col("_order_id_clean").isNull() | (F.col("_order_id_clean") == ""), "MISSING_ORDER_ID")
        .when(F.col("_pharmacy_id_clean").isNull() | (F.col("_pharmacy_id_clean") == ""), "MISSING_PHARMACY_ID")
        .when(F.col("_known_pharmacy_id").isNull(), "UNKNOWN_PHARMACY_ID")
        .when(F.col("order_time").isNull(), "INVALID_ORDER_TIME")
        .when(F.col("promised_delivery_time").isNull(), "INVALID_PROMISED_DELIVERY_TIME")
    )
)

order_cols = orders_bronze.columns
orders_accepted_raw = orders_checked.filter(F.col("validation_reason").isNull()).select(*order_cols)
orders_rejected = orders_checked.filter(F.col("validation_reason").isNotNull()).select(*order_cols, "validation_reason")

silver_orders_batch = orders_accepted_raw.select(
    F.trim("order_id").alias("order_id"),
    F.trim("pharmacy_id").alias("pharmacy_id"),
    F.col("order_time").cast("timestamp").alias("order_time"),
    F.col("promised_delivery_time").cast("timestamp").alias("promised_delivery_time"),
    "order_status", "_source_file", "_ingestion_timestamp", "_batch_id"
)

current_accepted_order_ids = silver_orders_batch.select("order_id").distinct()
valid_order_ids = existing_order_ids.unionByName(current_accepted_order_ids).distinct()

# -----------------------------------------------------------------
# 4. Validate CURRENT-BATCH order lines against prior + current orders.
# -----------------------------------------------------------------
lines_checked = (
    order_lines_bronze
    .withColumn("_order_id_clean", F.trim("order_id"))
    .withColumn("_line_id_clean", F.trim("line_id"))
    .withColumn("ordered_quantity", F.expr("try_cast(ordered_quantity AS BIGINT)"))
    .join(
        valid_order_ids.withColumnRenamed("order_id", "_valid_order_id"),
        F.col("_order_id_clean") == F.col("_valid_order_id"),
        "left",
    )
    .withColumn(
        "validation_reason",
        F.when(F.col("_order_id_clean").isNull() | (F.col("_order_id_clean") == ""), "MISSING_ORDER_ID")
        .when(F.col("_line_id_clean").isNull() | (F.col("_line_id_clean") == ""), "MISSING_LINE_ID")
        .when(F.col("ordered_quantity").isNull() | (F.col("ordered_quantity") <= 0), "INVALID_ORDERED_QUANTITY")
        .when(F.col("_valid_order_id").isNull(), "ORDER_NOT_ACCEPTED")
    )
)

line_cols = order_lines_bronze.columns
lines_accepted_raw = lines_checked.filter(F.col("validation_reason").isNull()).select(*line_cols)
lines_rejected = lines_checked.filter(F.col("validation_reason").isNotNull()).select(*line_cols, "validation_reason")

silver_order_lines_batch = lines_accepted_raw.select(
    F.trim("order_id").alias("order_id"),
    F.trim("line_id").alias("line_id"),
    "product_id",
    F.col("ordered_quantity").cast("long").alias("ordered_quantity"),
    "_source_file", "_ingestion_timestamp", "_batch_id"
)

current_order_line_keys = silver_order_lines_batch.select("order_id", "line_id").distinct()
valid_order_line_keys = existing_order_line_keys.unionByName(current_order_line_keys).distinct()

# -----------------------------------------------------------------
# 5. Validate CURRENT-BATCH shipment lines against prior + current order-lines.
# -----------------------------------------------------------------
shipments_checked = (
    shipment_lines_bronze
    .withColumn("_shipment_id_clean", F.trim("shipment_id"))
    .withColumn("_order_id_clean", F.trim("order_id"))
    .withColumn("_line_id_clean", F.trim("line_id"))
    .withColumn("shipped_quantity", F.expr("try_cast(shipped_quantity AS BIGINT)"))
    .join(
        valid_order_line_keys.select(
            F.col("order_id").alias("_valid_order_id"),
            F.col("line_id").alias("_valid_line_id"),
        ),
        (F.col("_order_id_clean") == F.col("_valid_order_id")) &
        (F.col("_line_id_clean") == F.col("_valid_line_id")),
        "left",
    )
    .withColumn(
        "validation_reason",
        F.when(F.col("_shipment_id_clean").isNull() | (F.col("_shipment_id_clean") == ""), "MISSING_SHIPMENT_ID")
        .when(F.col("_order_id_clean").isNull() | (F.col("_order_id_clean") == ""), "MISSING_ORDER_ID")
        .when(F.col("_line_id_clean").isNull() | (F.col("_line_id_clean") == ""), "MISSING_LINE_ID")
        .when(F.col("shipped_quantity").isNull() | (F.col("shipped_quantity") <= 0), "INVALID_SHIPPED_QUANTITY")
        .when(F.col("_valid_order_id").isNull(), "ORDER_LINE_NOT_ACCEPTED")
    )
)

shipment_cols = shipment_lines_bronze.columns
shipments_accepted_raw = shipments_checked.filter(F.col("validation_reason").isNull()).select(*shipment_cols)
# Existing notebook did not persist rejected shipment lines, so we preserve that behavior.
shipments_rejected = shipments_checked.filter(F.col("validation_reason").isNotNull()).select(*shipment_cols, "validation_reason")

silver_shipment_lines_batch = shipments_accepted_raw.select(
    F.trim("shipment_id").alias("shipment_id"),
    F.trim("order_id").alias("order_id"),
    F.trim("line_id").alias("line_id"),
    F.col("shipped_quantity").cast("long").alias("shipped_quantity"),
    "_source_file", "_ingestion_timestamp", "_batch_id"
)

current_shipment_keys = silver_shipment_lines_batch.select("shipment_id", "order_id", "line_id").distinct()
valid_shipment_keys = existing_shipment_keys.unionByName(current_shipment_keys).distinct()

# -----------------------------------------------------------------
# 6. Validate CURRENT-BATCH delivery-event fields and duplicate event IDs.
# Duplicate detection checks both within this batch and IDs already in Silver.
# -----------------------------------------------------------------
events_prepared = (
    delivery_events_bronze
    .withColumn("_event_id_clean", F.trim("event_id"))
    .withColumn("_shipment_id_clean", F.trim("shipment_id"))
    .withColumn("_order_id_clean", F.trim("order_id"))
    .withColumn("_line_id_clean", F.trim("line_id"))
    .withColumn("delivered_quantity", F.expr("try_cast(delivered_quantity AS BIGINT)"))
    .withColumn("delivery_time", F.expr("try_cast(delivery_time AS TIMESTAMP)"))
    .withColumn(
        "validation_reason",
        F.when(F.col("_event_id_clean").isNull() | (F.col("_event_id_clean") == ""), "MISSING_EVENT_ID")
        .when(F.col("_shipment_id_clean").isNull() | (F.col("_shipment_id_clean") == ""), "MISSING_SHIPMENT_ID")
        .when(F.col("_order_id_clean").isNull() | (F.col("_order_id_clean") == ""), "MISSING_ORDER_ID")
        .when(F.col("_line_id_clean").isNull() | (F.col("_line_id_clean") == ""), "MISSING_LINE_ID")
        .when(F.col("delivered_quantity").isNull() | (F.col("delivered_quantity") < 0), "INVALID_DELIVERED_QUANTITY")
        .when(F.col("delivery_time").isNull(), "INVALID_DELIVERY_TIME")
    )
)

event_cols = delivery_events_bronze.columns
events_field_rejected = events_prepared.filter(F.col("validation_reason").isNotNull()).select(*event_cols, "validation_reason")
events_field_valid = events_prepared.filter(F.col("validation_reason").isNull())

# Rank duplicates inside current batch.
dedup_window = Window.partitionBy("_event_id_clean").orderBy(
    F.col("_ingestion_timestamp").desc_nulls_last(),
    F.col("_source_file").desc_nulls_last(),
)
events_ranked = events_field_valid.withColumn("_event_rank", F.row_number().over(dedup_window))
within_batch_duplicates = (
    events_ranked.filter(F.col("_event_rank") > 1)
    .select(*event_cols)
    .withColumn("validation_reason", F.lit("DUPLICATE_EVENT_ID"))
)
events_one_per_id = events_ranked.filter(F.col("_event_rank") == 1).drop("_event_rank")

existing_event_ids = spark.table(SILVER["delivery_events"]).select(F.trim("event_id").alias("_existing_event_id")).distinct()
events_with_existing = events_one_per_id.join(
    existing_event_ids,
    F.col("_event_id_clean") == F.col("_existing_event_id"),
    "left",
)
already_processed_duplicates = (
    events_with_existing.filter(F.col("_existing_event_id").isNotNull())
    .select(*event_cols)
    .withColumn("validation_reason", F.lit("DUPLICATE_EVENT_ID"))
)
events_new = events_with_existing.filter(F.col("_existing_event_id").isNull()).drop("_existing_event_id")
events_duplicates = within_batch_duplicates.unionByName(already_processed_duplicates)

# Referencing a known shipment-line may use either an earlier Silver shipment or one accepted above.
events_matched = (
    events_new
    .join(
        valid_shipment_keys.select(
            F.col("shipment_id").alias("_valid_shipment_id"),
            F.col("order_id").alias("_valid_order_id"),
            F.col("line_id").alias("_valid_line_id"),
        ),
        (F.col("_shipment_id_clean") == F.col("_valid_shipment_id")) &
        (F.col("_order_id_clean") == F.col("_valid_order_id")) &
        (F.col("_line_id_clean") == F.col("_valid_line_id")),
        "left",
    )
)

events_accepted_base = events_matched.filter(F.col("_valid_shipment_id").isNotNull()).select(*event_cols)
events_quarantine = (
    events_matched.filter(F.col("_valid_shipment_id").isNull())
    .select(*event_cols)
    .withColumn("validation_reason", F.lit("UNKNOWN_SHIPMENT_LINE"))
)

# -----------------------------------------------------------------
# 7. Delivery exception flags.
# IMPORTANT incremental correctness:
# For affected order-lines we aggregate EXISTING Silver accepted events +
# the NEW accepted batch, instead of rescanning all Bronze delivery events.
# -----------------------------------------------------------------
affected_line_keys = events_accepted_base.select(
    F.trim("order_id").alias("order_id"),
    F.trim("line_id").alias("line_id"),
).distinct()

existing_events_for_affected_lines = (
    spark.table(SILVER["delivery_events"])
    .join(affected_line_keys, ["order_id", "line_id"], "inner")
    .select("event_id", "order_id", "line_id", "delivered_quantity")
)
new_events_for_totals = events_accepted_base.select(
    "event_id", "order_id", "line_id", "delivered_quantity"
)

delivered_totals = (
    existing_events_for_affected_lines
    .unionByName(new_events_for_totals)
    .groupBy("order_id", "line_id")
    .agg(F.sum(F.col("delivered_quantity").cast("long")).alias("total_delivered_quantity"))
)

# Order-line quantities/order creation time may come from previous Silver or this batch.
order_line_lookup = (
    spark.table(SILVER["order_lines"])
    .select("order_id", "line_id", F.col("ordered_quantity").cast("long").alias("ordered_quantity"))
    .unionByName(silver_order_lines_batch.select("order_id", "line_id", F.col("ordered_quantity").cast("long")))
    .dropDuplicates(["order_id", "line_id"])
)
order_lookup = (
    spark.table(SILVER["orders"])
    .select("order_id", F.col("order_time").cast("timestamp").alias("order_time"))
    .unionByName(silver_orders_batch.select("order_id", F.col("order_time").cast("timestamp")))
    .dropDuplicates(["order_id"])
)

silver_delivery_events_batch = (
    events_accepted_base
    .withColumn("order_id", F.trim("order_id"))
    .withColumn("line_id", F.trim("line_id"))
    .withColumn("shipment_id", F.trim("shipment_id"))
    .withColumn("event_id", F.trim("event_id"))
    .withColumn("delivery_time", F.col("delivery_time").cast("timestamp"))
    .join(order_line_lookup, ["order_id", "line_id"], "left")
    .join(order_lookup, ["order_id"], "left")
    .join(delivered_totals, ["order_id", "line_id"], "left")
    .withColumn("is_overdelivery", F.col("total_delivered_quantity") > F.col("ordered_quantity"))
    .withColumn("is_delivery_before_order", F.col("delivery_time") < F.col("order_time"))
    .select(
        "event_id", "shipment_id", "order_id", "line_id",
        F.col("delivered_quantity").cast("long").alias("delivered_quantity"),
        F.col("delivery_time").cast("timestamp").alias("delivery_time"),
        "is_overdelivery", "is_delivery_before_order",
        F.col("total_delivered_quantity").cast("long").alias("total_delivered_quantity"),
        "_source_file", "_ingestion_timestamp", "_batch_id"
    )
)

# -----------------------------------------------------------------
# 8. MERGE helpers.
# Accepted Silver is insert-only for this immutable-source design.
# Audit tables are also deduplicated by their stable business/audit keys.
# -----------------------------------------------------------------
def merge_insert_only(table_name, source_df, keys):
    if source_df.limit(1).count() == 0:
        print(f"{table_name}: nothing to merge")
        return 0

    source_df = source_df.dropDuplicates(keys)
    merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
    before = spark.table(table_name).count()
    (
        DeltaTable.forName(spark, table_name)
        .alias("t")
        .merge(source_df.alias("s"), merge_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )
    after = spark.table(table_name).count()
    print(f"{table_name}: inserted={after - before}, total={after}")
    return after - before

# For delivery events, cumulative flags can change on PRIOR events on the same
# affected line when a new event causes overdelivery. Update those affected rows only.
def refresh_affected_delivery_flags():
    if affected_line_keys.limit(1).count() == 0:
        return

    existing_affected = (
        spark.table(SILVER["delivery_events"])
        .join(affected_line_keys, ["order_id", "line_id"], "inner")
        .drop("total_delivered_quantity", "is_overdelivery")
        .join(delivered_totals, ["order_id", "line_id"], "left")
        .join(order_line_lookup, ["order_id", "line_id"], "left")
        .withColumn("is_overdelivery", F.col("total_delivered_quantity") > F.col("ordered_quantity"))
        .drop("ordered_quantity")
    )

    target = DeltaTable.forName(spark, SILVER["delivery_events"])
    (
        target.alias("t")
        .merge(existing_affected.alias("s"), "t.event_id = s.event_id")
        .whenMatchedUpdate(set={
            "total_delivered_quantity": "s.total_delivered_quantity",
            "is_overdelivery": "s.is_overdelivery",
        })
        .execute()
    )

# -----------------------------------------------------------------
# 9. Persist ONLY current-batch results.
# No overwrite of existing Silver tables.
# -----------------------------------------------------------------
print("\n=== CURRENT-BATCH VALIDATION COUNTS ===")
print("Accepted orders           :", silver_orders_batch.count())
print("Rejected orders           :", orders_rejected.count())
print("Accepted order lines      :", silver_order_lines_batch.count())
print("Rejected order lines      :", lines_rejected.count())
print("Accepted shipment lines   :", silver_shipment_lines_batch.count())
print("Rejected shipment lines   :", shipments_rejected.count())
print("Accepted delivery events  :", silver_delivery_events_batch.count())
print("Rejected delivery events  :", events_field_rejected.count())
print("Duplicate delivery events :", events_duplicates.count())
print("Quarantined events        :", events_quarantine.count())

merge_insert_only(SILVER["orders"], silver_orders_batch, ["order_id"])
merge_insert_only(SILVER["order_lines"], silver_order_lines_batch, ["order_id", "line_id"])
merge_insert_only(SILVER["shipment_lines"], silver_shipment_lines_batch, ["shipment_id", "order_id", "line_id"])
merge_insert_only(SILVER["delivery_events"], silver_delivery_events_batch, ["event_id"])

merge_insert_only(SILVER["rejected_orders"], orders_rejected, ["order_id", "_batch_id", "validation_reason"])
merge_insert_only(SILVER["rejected_order_lines"], lines_rejected, ["order_id", "line_id", "_batch_id", "validation_reason"])
merge_insert_only(SILVER["rejected_delivery_events"], events_field_rejected, ["event_id", "_batch_id", "validation_reason"])
merge_insert_only(SILVER["duplicate_delivery_events"], events_duplicates, ["event_id", "_batch_id", "validation_reason"])
merge_insert_only(SILVER["quarantined_delivery_events"], events_quarantine, ["event_id", "_batch_id", "validation_reason"])

refresh_affected_delivery_flags()

# Pharmacies are intentionally not rewritten because this Bronze incremental flow
# did not ingest new pharmacy rows. Existing silver_pharmacies remains the reference.

# -----------------------------------------------------------------
# 10. Verify only the processed batch + final target totals.
# -----------------------------------------------------------------
print("\n=== INCREMENTAL SILVER BATCH VERIFICATION ===")
for label, table in [
    ("orders", SILVER["orders"]),
    ("order_lines", SILVER["order_lines"]),
    ("shipment_lines", SILVER["shipment_lines"]),
    ("delivery_events", SILVER["delivery_events"]),
]:
    batch_count = spark.table(table).filter(F.col("_batch_id") == BATCH_ID).count()
    total_count = spark.table(table).count()
    print(f"{label}: batch={batch_count}, total={total_count}")

print("\nPHASE 4 INCREMENTAL SILVER JOB COMPLETED SUCCESSFULLY.")
print("Processed batch:", BATCH_ID)


Silver validation job started.
Catalog: dbwteam3
Schema: order_fulfillment
Input batch: all
Target: dbwteam3.order_fulfillment
